In [ ]:
%run ./01-config

In [ ]:
class Bronze:
    def __init__(self, env):
        conf = Config()
        self.landing_zone = conf.base_dir_data + '/raw'
        self.checkpoint_base  = conf.base_dir_checkpoint + '/checkpoints'
        self.catalog = env
        self.db_name = conf.db_name
        spark.sql(f'use {self.catalog}.{self.db_name}')

    def consume_user_registration(self, once=True, processing_time = '5 seconds'):
        from pyspark.sql import functions as f
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, LongType, DoubleType
        schema = (StructType([
                    StructField('user_id', StringType()),
                    StructField('device_id', LongType()),
                    StructField('mac_address', StringType()),
                    StructField('registration_timestamp', DoubleType())
                    ]))

        df_stream = (spark.readStream
                            .format('cloudFiles')
                            .schema(schema)
                            .option('cloudFiles.format', 'csv')
                            .option('header', True)
                            .option('maxFilePerTrigger', 1)
                            .load(self.landing_zone + '/registered_users_bz')
                            .withColumn('load_time', f.current_timestamp())
                            .withColumn('source_file', f.input_file_name())
                            )

        write_stream = (df_stream.writeStream
                                    .outputMode('append')
                                    .format('delta')
                                    .queryName('registered_users_bz_ingestion_stream')
                                    .option('checkpointLocation', self.checkpoint_base+'/registered_users_bz')
                            )
        
        spark.sparkContext.setLocalProperty("spark.scheduler.pool", "bronze_p2")

        if once:
            return write_stream.trigger(availableNow=True).toTable(f'{self.catalog}.{self.db_name}.registered_users_bz')
        else:
            return write_stream.trigger(processingTime= processing_time).toTable(f'{self.catalog}.{self.db_name}.registered_users_bz')
        
    def consume_gym_logins(self, once=True, processing_time='5 seconds'):
        from pyspark.sql import functions as f
        from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, LongType

        schema = StructType([
                            StructField('mac_address', StringType()),
                            StructField('gym', LongType()),
                            StructField('login', DoubleType()),
                            StructField('logout', DoubleType())
                                ])
        df_stream = (spark.readStream
                            .format('cloudFiles')
                            .schema(schema)
                            .option('cloudFiles.format', 'csv')
                            .option('header', True)
                            .option('maxFilesPerTrigger', 1)
                            .load(self.landing_zone+'/gym_logins_bz')
                            .withColumns({'load_time': f.current_timestamp(), 'source_file': f.input_file_name()})
                            )
        write_stream = (df_stream.writeStream
                                    .format('delta')
                                    .outputMode('append')
                                    .queryName('gym_logins_bz_ingestion_stream')
                                    .option('checkpointLocation', self.checkpoint_base + '/gym_logins_bz')
                                    )
        
        spark.sparkContext.setLocalProperty("spark.scheduler.pool", "bronze_p2")

        if once:
            return write_stream.trigger(availableNow=True).toTable(f'{self.catalog}.{self.db_name}.gym_logins_bz')
        else:
            return write_stream.trigger(processingTime=processing_time).toTable(f'{self.catalog}.{self.db_name}.gym_logins_bz')
        
    def consume_kafka_multiplex(self, once=True, processing_time='5 seconds'):
        from pyspark.sql import functions as f
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, LongType, DoubleType
        schema = StructType([
                            StructField('key', StringType()),
                            StructField('value', StringType()),
                            StructField('topic', StringType()),
                            StructField('partition', LongType()),
                            StructField('offset', LongType()),
                            StructField('timestamp', LongType())
                            ])
        df_date_lookup = spark.table(f'{self.catalog}.{self.db_name}.date_lookup').select('date', 'week_part')

        df_stream  = (spark.readStream
                            .format('cloudFiles')
                            .schema(schema)
                            .option('cloudFiles.format', 'json')
                            .load(self.landing_zone+'/kafka_multiplex_bz')
                            .withColumns({'load_time': f.current_timestamp(), 'source_file': f.input_file_name()})
                            .join(f.broadcast(df_date_lookup), 
                                  f.to_date((f.col('timestamp')/1000).cast(TimestampType())) == f.col('date'), 'left')
                            )
        write_stream = (df_stream.writeStream
                                    .format('delta')
                                    .outputMode('append')
                                    .queryName('kafka_multiplex_bz_ingestion_stream')
                                    .option('checkpointLocation', self.checkpoint_base +'/kafka_multiplex_bz')
                                    )
        
        spark.sparkContext.setLocalProperty("spark.scheduler.pool", "bronze_p1")
        
        if once:
            return write_stream.trigger(availableNow=True).toTable(f'{self.catalog}.{self.db_name}.kafka_multiplex_bz')
        else:
            return write_stream.trigger(processingTime=processing_time).toTable(f'{self.catalog}.{self.db_name}.kafka_multiplex_bz')
    
    def consume(self, once=True, processing_time = '5 seconds'):
        import time
        start = time.time()
        print('Starting bronze layer consumption')
        self.consume_user_registration(once, processing_time)
        self.consume_gym_logins(once, processing_time)
        self.consume_kafka_multiplex(once, processing_time)
        if once:
            for stream in spark.streams.active:
                stream.awaitTermination()
        print(f'Completed bronze layer consumption in {round(time.time() - start)} seconds')

    def assert_count(self, table_name, expected_count, filter='true'):
        actual_count = spark.table(f'{self.catalog}.{self.db_name}.{table_name}').where(filter).count()

        assert actual_count == expected_count, f'Expected {expected_count:,} records, but found {actual_count:,} in {table_name} where filter {filter}'
        print(f"Found {actual_count:,} / Expected {expected_count:,} records where {filter}: Success")

    def validate(self, sets):
        import time
        start = int(time.time())
        print(f"Validating bronze layer records...")
        self.assert_count('registered_users_bz', 5 if sets == 1 else 10)
        self.assert_count('gym_logins_bz', 8 if sets == 1 else 16)
        self.assert_count('kafka_multiplex_bz', 7 if sets == 1 else 13, 'topic = "user_info"')
        self.assert_count('kafka_multiplex_bz', 16 if sets == 1 else 32, 'topic = "workout"')
        self.assert_count('kafka_multiplex_bz', sets* 253801, 'topic = "bpm"')
        print(f"Bronze layer validation completed in {int(time.time()) - start} seconds")  
